In [1]:
import pandas as pd
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
df = pd.read_excel(r"C:\Users\Nandhana\OneDrive\Desktop\Internship\BraveEve\BraveEve_Content\NLP_dataset.xlsx")

df['label'] = df['label'].astype(str).str.strip().str.upper()
print(df.shape)
print(df['label'].value_counts())

(6440, 2)
label
YES    3365
NO     3075
Name: count, dtype: int64


In [3]:
# ----------------------------
# CLEANING FUNCTION
# ----------------------------
def clean_text(text):

    text = str(text).lower()

    contractions = {
        "i'm": "i am",
        "im": "i am",
        "don't": "do not",
        "cant": "cannot",
        "can't": "cannot",
        "won't": "will not",
        "isn't": "is not",
        "aren't": "are not",
        "wasn't": "was not",
        "weren't": "were not",
        "haven't": "have not",
        "hasn't": "has not",
        "hadn't": "had not",
        "shouldn't": "should not",
        "wouldn't": "would not",
        "couldn't": "could not",
        "didn't": "did not",
        "dont": "do not",
        "doesnt": "does not",
        "shouldnt": "should not",
        "ik": "i know",
        "theyre": "they are",
        "idk": "i do not know",
        "wanna": "want to",
        "gonna": "going to",
        "i'll": "i will"
    }

    for key, value in contractions.items():
        text = text.replace(key, value)

    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ----------------------------
# FEATURES & LABELS
# ----------------------------
X = df["text"]
y = df["label"]

In [4]:
# ----------------------------
# TRAIN TEST SPLIT
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

# ----------------------------
# NLP PIPELINE
# ----------------------------
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            stop_words=None,
            ngram_range=(1,3), 
            min_df=1,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

# ----------------------------
# TRAIN MODEL
# ----------------------------
model.fit(X_train, y_train)

# ----------------------------
# PREDICTIONS
# ----------------------------
y_pred = model.predict(X_test)

# ----------------------------
# EVALUATION
# ----------------------------
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ----------------------------
# SAVE MODEL
# ----------------------------
joblib.dump(model, "sentiment_model.pkl")
print("\nModel saved successfully!")


Accuracy: 0.9363354037267081

Classification Report:

              precision    recall  f1-score   support

          NO       0.93      0.94      0.93       308
         YES       0.94      0.93      0.94       336

    accuracy                           0.94       644
   macro avg       0.94      0.94      0.94       644
weighted avg       0.94      0.94      0.94       644


Model saved successfully!


In [5]:
# ----------------------------
# LOAD MODEL
# ----------------------------
model = joblib.load("sentiment_model.pkl")

In [6]:
# ----------------------------
# CLEAN FUNCTION
# ----------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ----------------------------
# TEST LOOP
# ----------------------------
while True:
    text = input("\nEnter patient response: ")
    if text.lower() == "exit":
        break
    cleaned = clean_text(text)
    prediction = model.predict([cleaned])[0]
    probabilities = model.predict_proba([cleaned])[0]
    labels = model.classes_
    confidence = dict(zip(labels, probabilities))
    print("\nPrediction:", prediction)
    print("\nConfidence Scores:")
    for label, prob in confidence.items():
        print(f"{label}: {prob:.2f}")


Prediction: NO

Confidence Scores:
NO: 0.60
YES: 0.40

Prediction: YES

Confidence Scores:
NO: 0.37
YES: 0.63


In [1]:
import os
os.getcwd()

'C:\\Users\\Nandhana\\BraveEve'